# 05 - Entrenamiento inicial de modelos para predicción de AKI

## Objetivo
El objetivo de este notebook fue entrenar los primeros modelos baseline para la predicción temprana de lesión renal aguda (`AKI_48h`) a partir del dataset final de modelado. Esta fase tuvo como propósito establecer una primera línea base de rendimiento, utilizando un flujo metodológicamente sólido, reproducible y compatible con las características del problema, especialmente el fuerte desbalanceo de la variable objetivo.

### 1. Importar librerías
Se importaron las librerías necesarias para:

- manipulación de datos (`pandas`, `numpy`)
- almacenamiento de modelos (`os`, `joblib`)
- partición del conjunto de datos (`train_test_split`)
- imputación de valores faltantes (`SimpleImputer`)
- construcción de pipelines y transformadores (`Pipeline`, `ColumnTransformer`)
- entrenamiento de modelos (`LogisticRegression`, `RandomForestClassifier`)
- evaluación de resultados (`roc_auc_score`, `precision_score`, `recall_score`, `f1_score`, `confusion_matrix`, `classification_report`)

La integración de estas herramientas en un único notebook permite construir un pipeline completo de modelado que combina preprocesamiento, entrenamiento y evaluación de manera reproducible con un enfoque estándar y robusto de aprendizaje automático supervisado en Python, adecuado para problemas clínicos de clasificación binaria.

In [4]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
FIGURES_DIR="../figures"
RESULTS_DIR="../results"

os.makedirs(FIGURES_DIR,exist_ok=True)
os.makedirs(RESULTS_DIR,exist_ok=True)

print("Carpetas de figures y results listas.")

Carpetas de figures y results listas.


### 2. Cargar el dataset
Se cargó el archivo `aki_dataset_model_ready.csv`, que contenía la versión final del conjunto de datos preparada para entrenamiento

In [5]:
file_path = "../data/processed/aki_dataset_model_ready.csv"

df = pd.read_csv(file_path)

print("Dataset final cargado correctamente.")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

Dataset final cargado correctamente.
Filas: 1175
Columnas: 14


### 3. Revisión rápida del dataset
Esta comprobación actúa como un control previo al entrenamiento y ayuda a prevenir errores derivados de archivos mal cargados o inconsistencias entre notebooks.


In [6]:
df.head()

,Sexo,EdadAproxIngreso,Edad_ge_65,Potasio_mmol_L,Sodio_mmol_L,BaselineCreatinina,AINE,IECA,ARAII,DIURETICO,VANCOMICINA,AMINOGLUCOSIDO,N_Farmacos_Riesgo,AKI_48h
0,0,61,0,4.83,144.0,0.96,0,0,0,0,0,0,0,0
1,0,46,0,4.41,142.0,1.15,0,0,0,0,0,0,0,0
2,1,25,0,4.09,138.0,0.75,1,0,0,0,0,0,1,0
3,0,78,1,4.70,143.0,0.97,0,0,0,0,0,0,0,0
4,1,35,0,4.43,140.0,0.63,0,0,0,0,0,0,0,0


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1175 entries, 0 to 1174
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Sexo                1175 non-null   int64  
 1   EdadAproxIngreso    1175 non-null   int64  
 2   Edad_ge_65          1175 non-null   int64  
 3   Potasio_mmol_L      1122 non-null   float64
 4   Sodio_mmol_L        1126 non-null   float64
 5   BaselineCreatinina  1173 non-null   float64
 6   AINE                1175 non-null   int64  
 7   IECA                1175 non-null   int64  
 8   ARAII               1175 non-null   int64  
 9   DIURETICO           1175 non-null   int64  
 10  VANCOMICINA         1175 non-null   int64  
 11  AMINOGLUCOSIDO      1175 non-null   int64  
 12  N_Farmacos_Riesgo   1175 non-null   int64  
 13  AKI_48h             1175 non-null   int64  
dtypes: float64(3), int64(11)
memory usage: 128.6 KB


In [8]:
df["AKI_48h"].value_counts(dropna=False)

AKI_48h
0    1146
1      29
Name: count, dtype: int64

### 4. Separación en predictores y variable objetivo
Se definió:
- `X` como el conjunto de predictores,
- `y` como la variable objetivo `AKI_48h`.

La separación entre predictores y target es el formato estándar requerido por los modelos supervisados de `scikit-learn`.

Este paso formaliza el problema como una tarea de clasificación binaria, donde el objetivo es aprender una función que relacione las variables clínicas de entrada con la probabilidad de desarrollar AKI en 48 horas.


In [9]:
target_col = "AKI_48h"

feature_cols = [col for col in df.columns if col != target_col]

X = df[feature_cols].copy()
y = df[target_col].copy()

print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

Dimensiones de X: (1175, 13)
Dimensiones de y: (1175,)


### 5. Train/test split estratificado
Se realizó una partición de los datos en conjuntos de entrenamiento y prueba utilizando:

- `test_size = 0.2`
- `random_state = 42`
- `stratify = y`

Dado que la variable objetivo está fuertemente desbalanceada, era imprescindible mantener una proporción similar de casos positivos y negativos tanto en entrenamiento como en prueba. La estratificación evita que uno de los subconjuntos quede desproporcionadamente pobre en casos positivos, lo que podría sesgar la evaluación.

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Dimensiones de X_train:", X_train.shape)
print("Dimensiones de X_test:", X_test.shape)
print("Dimensiones de y_train:", y_train.shape)
print("Dimensiones de y_test:", y_test.shape)

Dimensiones de X_train: (940, 13)
Dimensiones de X_test: (235, 13)
Dimensiones de y_train: (940,)
Dimensiones de y_test: (235,)


### 6. Revisión de distribución de clases en train y test
Se revisó la distribución de `AKI_48h` en ambos subconjuntos tras la partición.

**Resultados observados**
- En `y_train`: 917 negativos y 23 positivos
- En `y_test`: 229 negativos y 6 positivos

Las proporciones de clase fueron prácticamente idénticas al conjunto original.

El resultado confirmó que la partición fue adecuada y que los modelos serían entrenados y evaluados sobre subconjuntos consistentes con la naturaleza desbalanceada del problema original.

In [11]:
print("Distribución de clases en y_train:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True) * 100)

print("\nDistribución de clases en y_test:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True) * 100)

Distribución de clases en y_train:
AKI_48h
0    917
1     23
Name: count, dtype: int64
AKI_48h
0    97.553191
1     2.446809
Name: proportion, dtype: float64

Distribución de clases en y_test:
AKI_48h
0    229
1      6
Name: count, dtype: int64
AKI_48h
0    97.446809
1     2.553191
Name: proportion, dtype: float64


### 7. Definición de variables númericas y binarias
Se separaron los predictores en dos grupos:

**Variables numéricas continuas**
- `Potasio_mmol_L`
- `Sodio_mmol_L`
- `BaselineCreatinina`

**Variables binarias o enteras**
- `Sexo`
- `EdadAproxIngreso`
- `Edad_ge_65`
- `AINE`
- `IECA`
- `ARAII`
- `DIURETICO`
- `VANCOMICINA`
- `AMINOGLUCOSIDO`
- `N_Farmacos_Riesgo`

Esta clasificación fue necesaria para aplicar un tratamiento diferenciado de los valores faltantes y construir un preprocesamiento ordenado dentro del pipeline.

El conjunto final de variables incluía tanto biomarcadores continuos como variables clínicas binarias y derivadas, configurando una representación mixta pero manejable del estado del paciente.



In [12]:
numeric_features = [
    "Potasio_mmol_L",
    "Sodio_mmol_L",
    "BaselineCreatinina"
]

binary_or_integer_features = [
    "Sexo",
    "EdadAproxIngreso",
    "Edad_ge_65",
    "AINE",
    "IECA",
    "ARAII",
    "DIURETICO",
    "VANCOMICINA",
    "AMINOGLUCOSIDO",
    "N_Farmacos_Riesgo"
]

### 8. Creación del preprocesamiento
Se definió un transformador para las variables numéricas, basado en imputación por mediana, y un `ColumnTransformer` que:

- imputaba las variables continuas con missing
- dejaba pasar las variables binarias/enteras sin modificación.

Los valores faltantes residuales eran pocos y se concentraban en variables continuas. La mediana se consideró una estrategia prudente, sencilla y robusta frente a posibles valores extremos. Además, al incorporar la imputación dentro del pipeline, se garantizó que el ajuste del imputador se realizara únicamente con los datos de entrenamiento.

Este paso fue esencial para evitar fuga de información durante el preprocesamiento y asegurar que el tratamiento de missing fuera compatible con una evaluación honesta del modelo.

In [13]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("bin", "passthrough", binary_or_integer_features)
    ]
)

### 9. Pipeline de regresión logística
Se creó un pipeline compuesto por:
- el preprocesador definido previamente
- y un modelo `LogisticRegression` con:
  - `class_weight = "balanced"`
  - `max_iter = 1000`
  - `random_state = 42`

La regresión logística fue seleccionada como modelo baseline interpretable. El uso de `class_weight="balanced"` respondió al fuerte desbalanceo del problema y buscó compensar el peso mucho mayor de la clase negativa. El aumento de `max_iter` garantizó convergencia adecuada.

Este pipeline representa una línea base clínicamente defendible, transparente e interpretable, muy útil como referencia en un problema sanitario.

In [14]:
logreg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

### 10. Entrenamiento de regresión logística
Se ajustó el pipeline de regresión logística sobre `X_train` e `y_train`.

El entrenamiento en el subconjunto de entrenamiento permite que el modelo aprenda relaciones entre predictores y outcome sin utilizar información del conjunto de prueba.

Este paso dio lugar al primer modelo baseline del proyecto, que posteriormente se comparó con un modelo no lineal.


In [15]:
logreg_pipeline.fit(X_train, y_train)

print("Regresión logística entrenada correctamente.")

Regresión logística entrenada correctamente.


### 11. Predicciones de regresión logística
Se obtuvieron:
- las clases predichas (`y_pred_logreg`)
- y las probabilidades estimadas para la clase positiva (`y_proba_logreg`)

Las clases predichas son necesarias para métricas como recall, precision, F1 y matriz de confusión, mientras que las probabilidades son necesarias para ROC-AUC.

Esta distinción es importante porque permite evaluar el modelo tanto como clasificador binario con umbral por defecto como en términos de su capacidad de ranking probabilístico.

In [16]:
y_pred_logreg = logreg_pipeline.predict(X_test)
y_proba_logreg = logreg_pipeline.predict_proba(X_test)[:, 1]

### 12. Primera evaluación de la regresión logística

In [17]:
roc_auc_logreg = roc_auc_score(y_test, y_proba_logreg)
precision_logreg = precision_score(y_test, y_pred_logreg, zero_division=0)
recall_logreg = recall_score(y_test, y_pred_logreg, zero_division=0)
f1_logreg = f1_score(y_test, y_pred_logreg, zero_division=0)
cm_logreg = confusion_matrix(y_test, y_pred_logreg)

print("=== Regresión Logística ===")
print("ROC-AUC:", round(roc_auc_logreg, 4))
print("Precision:", round(precision_logreg, 4))
print("Recall:", round(recall_logreg, 4))
print("F1-score:", round(f1_logreg, 4))
print("Matriz de confusión:")
print(cm_logreg)

print("\nClassification report:")
print(classification_report(y_test, y_pred_logreg, zero_division=0))

=== Regresión Logística ===
ROC-AUC: 0.7475
Precision: 0.0606
Recall: 0.6667
F1-score: 0.1111
Matriz de confusión:
[[167  62]
 [  2   4]]

Classification report:
              precision    recall  f1-score   support

           0       0.99      0.73      0.84       229
           1       0.06      0.67      0.11         6

    accuracy                           0.73       235
   macro avg       0.52      0.70      0.48       235
weighted avg       0.96      0.73      0.82       235



Se calcularon las siguientes métricas en el conjunto de prueba:

- ROC-AUC: **0.7475**
- Precision: **0.0606**
- Recall: **0.6667**
- F1-score: **0.1111**

La matriz de confusión fue:

- TN = 167
- FP = 62
- FN = 2
- TP = 4

Estas métricas fueron seleccionadas porque resultan más informativas que la accuracy en un problema fuertemente desbalanceado.

La regresión logística logró identificar 4 de los 6 casos positivos del conjunto de prueba, lo que se traduce en un recall del 66.7%. Este resultado sugiere una buena sensibilidad relativa para un modelo baseline. No obstante, esta capacidad se obtuvo a costa de un número elevado de falsos positivos, lo que redujo mucho la precisión.

Desde el punto de vista clínico, el modelo se comportó como un sistema más sensible que específico, útil potencialmente como herramienta de alerta o cribado, aunque todavía con una tasa elevada de alarmas falsas.

### 13. Pipeline de Random Forest
Se definió un segundo pipeline, compuesto por:
- el mismo preprocesamiento anterior,
- y un `RandomForestClassifier` con:
  - `n_estimators = 200`
  - `class_weight = "balanced"`
  - `random_state = 42`

El Random Forest se incluyó como modelo de comparación no lineal. Este tipo de algoritmo puede captar relaciones más complejas entre variables y suele ser una referencia útil frente a modelos lineales como la regresión logística.

La inclusión de este segundo modelo permitió comparar dos enfoques distintos:
- uno lineal e interpretable
- otro basado en ensamble y potencialmente más flexible.

In [18]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42
    ))
])

### 14. Entrenamiento del Random Forest
Se ajustó el pipeline de Random Forest con el subconjunto de entrenamiento.

Al igual que en la regresión logística, el entrenamiento se realizó únicamente sobre `X_train` e `y_train` para preservar la validez del conjunto de prueba como evaluación independiente.

Este paso generó el segundo modelo baseline del proyecto, orientado a explorar si una aproximación no lineal mejoraba la capacidad de discriminación del sistema.


In [19]:
rf_pipeline.fit(X_train, y_train)

print("Random Forest entrenado correctamente.")

Random Forest entrenado correctamente.


### 15. Predicciones del Random Forest
Se obtuvieron:
- las clases predichas (`y_pred_rf`)
- y las probabilidades estimadas (`y_proba_rf`)

Nuevamente, esta separación fue necesaria para evaluar tanto el comportamiento binario del clasificador como su capacidad de ranking probabilístico.

In [20]:
y_pred_rf = rf_pipeline.predict(X_test)
y_proba_rf = rf_pipeline.predict_proba(X_test)[:, 1]

### 16. Primera evaluación del Random Forest

In [21]:
roc_auc_rf = roc_auc_score(y_test, y_proba_rf)
precision_rf = precision_score(y_test, y_pred_rf, zero_division=0)
recall_rf = recall_score(y_test, y_pred_rf, zero_division=0)
f1_rf = f1_score(y_test, y_pred_rf, zero_division=0)
cm_rf = confusion_matrix(y_test, y_pred_rf)

print("=== Random Forest ===")
print("ROC-AUC:", round(roc_auc_rf, 4))
print("Precision:", round(precision_rf, 4))
print("Recall:", round(recall_rf, 4))
print("F1-score:", round(f1_rf, 4))
print("Matriz de confusión:")
print(cm_rf)

print("\nClassification report:")
print(classification_report(y_test, y_pred_rf, zero_division=0))

=== Random Forest ===
ROC-AUC: 0.8952
Precision: 0.0
Recall: 0.0
F1-score: 0.0
Matriz de confusión:
[[228   1]
 [  6   0]]

Classification report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       229
           1       0.00      0.00      0.00         6

    accuracy                           0.97       235
   macro avg       0.49      0.50      0.49       235
weighted avg       0.95      0.97      0.96       235



Se calcularon las métricas del Random Forest en el conjunto de prueba:

- ROC-AUC: **0.8952**
- Precision: **0.0000**
- Recall: **0.0000**
- F1-score: **0.0000**

La matriz de confusión fue:

- TN = 228
- FP = 1
- FN = 6
- TP = 0

Estas métricas permitieron comparar de forma homogénea el rendimiento del Random Forest frente a la regresión logística.

El resultado del Random Forest fue especialmente relevante desde el punto de vista metodológico. A pesar de alcanzar un ROC-AUC muy superior al de la regresión logística, el modelo no identificó ningún caso positivo utilizando el umbral estándar de 0.5. Esto provocó un recall y una precisión iguales a cero.

Este comportamiento indica que el Random Forest presentaba una muy buena capacidad de discriminación global en términos de ordenación probabilística, pero que su decisión binaria bajo el umbral por defecto era excesivamente conservadora en este problema. En otras palabras, el modelo parecía distinguir razonablemente bien entre mayor y menor riesgo, pero no asignaba probabilidades suficientemente altas como para clasificar casos positivos en el test.

Este hallazgo es frecuente en datasets fuertemente desbalanceados y sugiere que el rendimiento clínico del modelo podría mejorar mediante un ajuste posterior del umbral de clasificación.

### 17. Resumen comparativo inicial de métricas
Se consolidaron en una tabla las métricas principales de ambos modelos:

- Regresión Logística
- Random Forest

**Resultados comparativos**
- El **Random Forest** obtuvo el mejor **ROC-AUC**
- La **Regresión Logística** obtuvo mejor **recall** y mejor **F1** en la clase positiva bajo el umbral estándar

La comparación estructurada de métricas permite evitar conclusiones simplistas sobre qué modelo es “mejor” y obliga a considerar qué criterio de rendimiento resulta más importante en el contexto clínico.

Los resultados muestran que la elección del mejor modelo depende de la métrica priorizada:
- si se prioriza la capacidad global de discriminación, el Random Forest parece superior
- si se prioriza la sensibilidad clínica para detectar AKI, la regresión logística resulta más útil en esta fase baseline.

In [22]:
results_df = pd.DataFrame({
    "Modelo": ["Regresión Logística", "Random Forest"],
    "ROC_AUC": [roc_auc_logreg, roc_auc_rf],
    "Precision": [precision_logreg, precision_rf],
    "Recall": [recall_logreg, recall_rf],
    "F1_score": [f1_logreg, f1_rf]
})
results_df.to_csv(f"{RESULTS_DIR}/results_05_model_results_comparison.csv", index=False)
results_df.sort_values(by="ROC_AUC", ascending=False)

,Modelo,ROC_AUC,Precision,Recall,F1_score
1,Random Forest,0.895197,0.000000,0.000000,0.000000
0,Regresión Logística,0.747453,0.060606,0.666667,0.111111


### 18. Guardado de modelos entrenados
Se almacenaron ambos pipelines entrenados en la carpeta `models/`:

- `logreg_pipeline.pkl`
- `rf_pipeline.pkl`


In [21]:
os.makedirs("../models", exist_ok=True)

joblib.dump(logreg_pipeline, "../models/logreg_pipeline.pkl")
joblib.dump(rf_pipeline, "../models/rf_pipeline.pkl")

print("Modelos guardados correctamente en la carpeta models.")

Modelos guardados correctamente en la carpeta models.


In [23]:
cm_logreg_df = pd.DataFrame(cm_logreg, 
                            index=["True_0", "True_1"], 
                            columns=["Pred_0", "Pred_1"])
cm_rf_df = pd.DataFrame(cm_rf, 
                        index=["True_0", "True_1"], 
                        columns=["Pred_0", "Pred_1"])

cm_logreg_df.to_csv(f"{RESULTS_DIR}/results_05_confusion_matrix_logreg.csv")
cm_rf_df.to_csv(f"{RESULTS_DIR}/results_05_confusion_matrix_rf.csv")

In [24]:
report_logreg = classification_report(y_test, y_pred_logreg, zero_division=0, output_dict=True)
report_rf = classification_report(y_test, y_pred_rf, zero_division=0, output_dict=True)

pd.DataFrame(report_logreg).T.to_csv(f"{RESULTS_DIR}/results_05_classification_report_logreg.csv")
pd.DataFrame(report_rf).T.to_csv(f"{RESULTS_DIR}/results_05_classification_report_rf.csv")

### 19. Conclusiones del notebook 

En este notebook se entrenaron dos modelos baseline para la predicción de AKI:

- una regresión logística, como modelo interpretable de referencia,
- y un random forest, como modelo no lineal de comparación.

Las principales decisiones metodológicas fueron:

- la partición estratificada preservó correctamente la distribución del target
- la imputación integrada en el pipeline permitió un tratamiento reproducible de los valores faltantes
- la regresión logística mostró una mejor capacidad para detectar casos positivos
- el Random Forest presentó una mejor capacidad global de discriminación según ROC-AUC, pero no detectó ningún caso positivo con el umbral estándar de 0.5

El resultado más importante de esta fase no fue únicamente la comparación entre modelos, sino la identificación de dos comportamientos distintos y complementarios:
- un modelo lineal más sensible
- y un modelo no lineal con mejor ranking probabilístico, pero peor clasificación positiva bajo el umbral por defecto

Este notebook establece la primera línea base del rendimiento predictivo del proyecto y deja preparados tanto los modelos como el conjunto de resultados para una evaluación más detallada en el notebook siguiente.